# UKHousing: matched inducing budgets

Select a GPU runtime and Run all. Models/data load from latest main;
no implementations are embedded. Setup can be rerun without reconnecting.
Every package-dependent stage runs in a fresh worker.

SVGP 1,024; GPW and ProSpar each 8 x 128 and 4 x 256; VNNGP K=32/128
with one fixed inducing location per training observation. Standard
Patchwork: checkerboard 4 x 2, GPW-center Voronoi and 8-leaf PC tree,
each using the 8-patch GPW initialization's 1,024 observed targets.
These are reduced-data baselines, not equal-information comparisons.

Approved protocol: ARD RBF, float64, 100 epochs, 20 kernel/noise warm-up
for variational models. Standard Patchwork uses 100 joint-likelihood
updates without warm-up. Other existing UKHousing settings are retained.
ProSpar starts with temperatures=0.1, exponent=2, nuggets=0.01, prior_jitter=0.

The 2018-flats reconstruction is not the verified VNNGP snapshot.
This is NOT a reproduction: that paper used Matern-5/2, other initial
values, optimizer sweeps and three seeds. Here seed=31, split=64/16/20,
training-only standardization of log prices, fixed geographic ROI.
Source/data checksums, full configurations, checkpoints, predictions,
histories, metrics and maps are archived per completed run.
Failures stop downstream execution; no recovery or device substitution.


## 1. Source and dependencies


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['checkout', 'install', 'settings', 'data', 'training', 'download']:
    _OISST_STAGES[_stage] = False
for _required in []:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/content/gpw_ukhousing_comparison')
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if ROOT.exists():
    top = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--show-toplevel"], text=True).strip()
    if Path(top).resolve() != ROOT.resolve():
        raise ValueError("Existing checkout must be its own Git repository")
    remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    if remote != REPOSITORY or branch != BRANCH:
        raise ValueError("Existing checkout repository or branch does not match the requested source")
    subprocess.run(["git", "-C", str(ROOT), "diff", "--exit-code", "HEAD"], check=True)
    untracked = subprocess.check_output(["git", "-C", str(ROOT), "ls-files", "--others",
        "--exclude-standard", "--", ":(glob)src/**/*.py", ":(glob)experiments/**/*.py"], text=True).strip()
    if untracked:
        raise ValueError(f"Untracked source files prevent verified reuse: {untracked}")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge-base", "--is-ancestor", "HEAD", "origin/" + BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH,
                    "--single-branch", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)

COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
for path in reversed((str(ROOT / "src"), str(ROOT))):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_OISST_STAGES['checkout'] = True
_GPW_CELL_RESULT


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['install', 'settings', 'data', 'training', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['checkout']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--editable", str(ROOT) + "[ukhousing]"], check=True)
import runpy
run_worker = runpy.run_path(str(ROOT / "experiments/colab_client.py"))["run_worker"]

def call_worker(action, *, on_event=None, **values):
    return run_worker(ROOT, {"action": action, "source_provenance": SOURCE_PROVENANCE, **values},
                      module="experiments.ukhousing_worker", on_event=on_event)
_OISST_STAGES['install'] = True


## 2. Select models and override hyperparameters

GPW denotes 8 x 128. Remove names to select fewer runs; order is preserved.
All effective fields are displayed and saved. Example:
OVERRIDES = {"GPW": {"epochs": 50, "warmup_epochs": 10}}.
Baselines reference GPW even when it is deselected, not GPW 4 x 256.
ProSpar overrides use an explicit prospar dictionary with all four fields.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['settings', 'training', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
from datetime import UTC, datetime
from IPython.display import Image, Markdown, display
from google.colab import files
import pandas as pd

MODELS = ["SVGP", "GPW", "GPW 4 x 256", "ProSpar 8 x 128", "ProSpar 4 x 256",
          "VNNGP K32", "VNNGP K128", "Standard Patchwork",
          "Standard Patchwork (GPW regions)", "Standard Patchwork (PC tree)"]
OVERRIDES = {}
DATA = {"seed": 31, "train_fraction": 0.64, "validation_fraction": 0.16}
settings = call_worker("settings", device="cuda:0", models=MODELS, overrides=OVERRIDES)
RUNS = settings["runs"]
WORK_ROOT = ROOT / "results/ukhousing_comparison"
_GPW_CELL_RESULT = display(pd.DataFrame([{"run": row["name"], **row["config"]} for row in RUNS]))
_OISST_STAGES['settings'] = True
_GPW_CELL_RESULT


## 3. Download and prepare data once

Keep this stage unchanged when only model settings change. Training
verifies prepared configuration/checksums; it never silently rebuilds data.
Rerun this cell explicitly to prepare a new immutable split directory.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['data', 'training', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREPARED_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("prepared_%Y%m%dT%H%M%S%fZ")
prepared = call_worker("prepare", data=DATA, directory=str(PREPARED_DIRECTORY))
_GPW_CELL_RESULT = print("Prepared:", prepared["directory"])
_OISST_STAGES['data'] = True
_GPW_CELL_RESULT


## 4. Train selected configurations

Edit MODELS/OVERRIDES and rerun settings, then this cell, to run only the
selected configurations on the verified split. Each invocation trains
from scratch in a new batch. Completed runs remain available after a failure.
One live plot updates in place; final maps and metrics display per run.
Worker exit releases GPU memory before the next notebook action; the
existing runner also releases each model/cache before its archive event.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['training', 'download']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'data']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
live = display("Starting UKHousing comparison...", display_id=True)

def show_event(event):
    path = Path(event["directory"])
    if event["kind"] == "snapshot":
        live.update(Image(filename=str(path / "progress.png")))
    elif event["kind"] == "complete":
        display(Markdown("### " + event["name"]))
        display(pd.DataFrame([event["metrics"]]).T)
        display(Image(filename=str(path / "maps.png")))
        files.download(event["archive"])
    else:
        raise ValueError(f"Unknown worker event: {event['kind']}")

BATCH_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("batch_%Y%m%dT%H%M%S%fZ")
completed = call_worker("train", data=DATA, runs=RUNS, prepared_directory=str(PREPARED_DIRECTORY),
                        directory=str(BATCH_DIRECTORY), on_event=show_event)
display(pd.read_csv(Path(completed["directory"]) / "comparison.csv"))
_GPW_CELL_RESULT = print("Results:", completed["directory"])
_OISST_STAGES['training'] = True
_GPW_CELL_RESULT


## 5. Download complete batch


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['download']:
    _OISST_STAGES[_stage] = False
for _required in ['training']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import shutil
archive = shutil.make_archive(str(BATCH_DIRECTORY), "zip", root_dir=BATCH_DIRECTORY)
_GPW_CELL_RESULT = files.download(archive)
_OISST_STAGES['download'] = True
_GPW_CELL_RESULT


## References
- [VNNGP paper](https://proceedings.mlr.press/v162/wu22h.html)
- [Patchwork Kriging](https://jmlr.org/papers/v19/17-042.html)
- [ProSpar derivation](https://arxiv.org/abs/2311.08752v1)
- [UKHousing preparation reference](https://github.com/cunningham-lab/hipgp/blob/master/experiments-hip-gp/uk_housing_data.py)
